For the BASELINE EXPERIMENTATION I AM GOING TO PICK AT LEAST THREE WITHOUT ANY HYPER PARAMETER TUNING

In [1]:
#import dependencies
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
import mlflow
from sklearn.metrics import accuracy_score,classification_report

C:\Users\siawc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the dataset
file_path = r'C:\Users\siawc\OneDrive\Desktop\Felix\mental health\eda& preprocessing\health_folder\mental_heath_unbanlanced_cleaned.csv'

data = pd.read_csv(file_path)
data.head()

,status,cleaned_text
0,Anxiety,oh gosh
1,Anxiety,trouble sleeping confused mind restless heart ...
2,Anxiety,wrong back dear forward doubt stay restless re...
3,Anxiety,ive shifted focus something else im still worried
4,Anxiety,im restless restless month boy mean


In [3]:
#checking for null values
data.isnull().sum()

status           0
cleaned_text    78
dtype: int64

In [4]:
# dropping nan values
data.dropna(inplace=True)

In [5]:
data.isnull().sum()

status          0
cleaned_text    0
dtype: int64

In [6]:
#encoding the status column
data['status'].value_counts()

status
Normal        18317
Depression    14505
Suicidal      11209
Anxiety        5503
Name: count, dtype: int64

In [7]:
encoder = LabelEncoder()
data['status'] = encoder.fit_transform(data['status'])

data['status'].value_counts()

status
2    18317
1    14505
3    11209
0     5503
Name: count, dtype: int64

In [8]:
# splitting the data into train,test and evaluation data
X = data.drop('status',axis=1)
y = data['status']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

X_eval,X_test,y_eval,y_test = train_test_split(X_test,y_test, test_size=0.5,random_state=42)

In [9]:
#shaping of the training,testing and evaluation data
print(f'Shape of X_train is: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Shape of X_test is: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Shape of X_eval is: {X_eval.shape[0]} ({X_eval.shape[0]/len(X)*100:.1f}%)')


Shape of X_train is: 34673 (70.0%)
Shape of X_test is: 7431 (15.0%)
Shape of X_eval is: 7430 (15.0%)


In [10]:
def convert_words_to_vec(X_train, X_test, mode='tfidf'):
    """
    Vectorize training and test sets using the specified vectorizer.

    Parameters
    ----------
    X_train : pandas Series or DataFrame
    X_test  : pandas Series or DataFrame
    mode    : str, either 'tfidf' or 'count'

    Returns
    -------
    X_train_vec, X_test_vec, vectorizer
    """
    if mode == 'tfidf':
        vectorizer = TfidfVectorizer()
    else:
        vectorizer = CountVectorizer()

    # make sure we pass a flat array of text values
    X_train_text = X_train.values.ravel()
    X_test_text = X_test.values.ravel()

    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_test_vec = vectorizer.transform(X_test_text)
    return X_train_vec, X_test_vec, vectorizer


In [11]:
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('Mental heath Category')

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1772713985682, experiment_id='2', last_update_time=1772713985682, lifecycle_stage='active', name='Mental heath Category', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [12]:
def train_eval_test(X_train, y_train, X_test, y_test, X_eval, y_eval, mode='tfidf'):
    """
    Convert the raw text data to vectors, train a handful of baseline
    classifiers and evaluate them on the held‑out splits.  Metrics are
    logged to MLflow for easy comparison.

    Parameters
    ----------
    X_train, X_test, X_eval : pandas Series/DataFrame of text
    y_train, y_test, y_eval : array-like labels
    mode : str, either 'tfidf' (default) or 'count' to choose vectorizer
    """
    # vectorization
    X_train_vec, X_test_vec, vectorizer = convert_words_to_vec(X_train, X_test, mode)
    X_eval_vec = vectorizer.transform(X_eval.values.ravel())

    models = {
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "RandomForest": RandomForestClassifier(),
        "SVM": SVC(),
        "GaussianNB": GaussianNB()
    }

    for name, model in models.items():
        print(f"\n Training {name} ({mode})")
        with mlflow.start_run(run_name=f"{name}_{mode}"):
            model.fit(X_train_vec, y_train)

            test_pred = model.predict(X_test_vec)
            eval_pred = model.predict(X_eval_vec)

            test_acc = accuracy_score(y_test,test_pred)
            eval_acc = accuracy_score(y_eval,eval_pred)


            test_report = classification_report(y_test,test_pred,output_dict=True)
            eval_report = classification_report(y_eval,eval_pred, output_dict=True)

            for label,metrics in test_report.items():
                if isinstance(metrics,dict):
                    for metric,value in metrics.items():
                        mlflow.log_metric(f'test_{label}_{metric}',value)
            
            for label,metrics in eval_report.items():
                if isinstance(metrics,dict):
                    for metric,value in metrics.items():
                        mlflow.log_metric(f'eval_{label}_{metric}',value)



            # log parameters/metrics so we can review them later
            mlflow.log_param("vectorizer", mode)
            mlflow.log_metric("test_accuracy", test_acc)
            mlflow.log_metric("eval_accuracy", eval_acc)

            print(f"Test accuracy: {test_acc:.4f}")
            print(f"Eval accuracy: {eval_acc:.4f}")

    print("Training complete.")


In [13]:
# run the training routine with both vectorizers
for vect in ['tfidf', 'count']:
    print(f"\n===== Vectorizer: {vect} =====")
    train_eval_test(X_train, y_train, X_test, y_test, X_eval, y_eval, mode=vect)



===== Vectorizer: tfidf =====

 Training LogisticRegression (tfidf)
Test accuracy: 0.7781
Eval accuracy: 0.7813
🏃 View run LogisticRegression_tfidf at: http://127.0.0.1:5000/#/experiments/2/runs/93c8d96b35e0494e97555695ff63c9c4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

 Training RandomForest (tfidf)
Test accuracy: 0.7261
Eval accuracy: 0.7295
🏃 View run RandomForest_tfidf at: http://127.0.0.1:5000/#/experiments/2/runs/96d11775ab7741f0a298211d6bc6333d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

 Training SVM (tfidf)
Test accuracy: 0.7844
Eval accuracy: 0.7919
🏃 View run SVM_tfidf at: http://127.0.0.1:5000/#/experiments/2/runs/4b94f2ca86fb4e3a9bd83446135ebc47
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

 Training GaussianNB (tfidf)
🏃 View run GaussianNB_tfidf at: http://127.0.0.1:5000/#/experiments/2/runs/183f130c98e248cd84b53e3f03410cfe
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


TypeError: Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.